In [3]:
# !pip install -q -U bitsandbytes peft trl transformers datasets

In [4]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print('VRAM cleared')
for i in range(torch.cuda.device_count()):
    total    = torch.cuda.get_device_properties(i).total_memory / 1e9
    reserved = torch.cuda.memory_reserved(i) / 1e9
    free     = total - reserved
    print(f'  GPU {i}: {free:.1f} GB free / {total:.1f} GB total')

 VRAM cleared
  GPU 0: 15.6 GB free / 15.6 GB total
  GPU 1: 15.6 GB free / 15.6 GB total


In [5]:
import os
import json
import torch
import bitsandbytes as bnb
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import DPOTrainer, DPOConfig
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from transformers.trainer_utils import get_last_checkpoint
from transformers import EarlyStoppingCallback

In [ ]:
# config setup

DPO_DATASET_PATH = '/kaggle/input/datasets/<<kaggle_username>>/llm-finetune-dpo-dataset'

DPO_TRAIN_FILE = f'{DPO_DATASET_PATH}/dpo_train_clean.jsonl'
DPO_VAL_FILE   = f'{DPO_DATASET_PATH}/dpo_val_clean.jsonl'
DPO_TEST_FILE  = f'{DPO_DATASET_PATH}/dpo_test_clean.jsonl'

SFT_ADAPTER = "/kaggle/input/datasets/<<kaggle_username>>/sft-lora/kaggle/working/sft_lora/checkpoint-303"  # ← CHANGE THIS

# DPO output directory (stays in Kaggle working dir)
DPO_DIR = '/kaggle/working/dpo_lora'
os.makedirs(DPO_DIR, exist_ok=True)

# Base model
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'

# ── Verify files exist ──
print(' Verifying file paths...')
checks = {
    'DPO train':  DPO_TRAIN_FILE,
    'DPO val':    DPO_VAL_FILE,
    'DPO test':   DPO_TEST_FILE,
    'SFT adapter': SFT_ADAPTER,
}
all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    icon   = '' if exists else ' NOT FOUND'
    print(f'  {icon}  {label}: {path}')
    if not exists:
        all_ok = False

if not all_ok:
    print('\n  Fix the missing paths above before continuing!')
else:
    print('\n All paths verified!')

 Verifying file paths...
    DPO train: /kaggle/input/datasets/ganeshiims/llm-finetune-dpo-dataset/dpo_train_clean.jsonl
    DPO val: /kaggle/input/datasets/ganeshiims/llm-finetune-dpo-dataset/dpo_val_clean.jsonl
    DPO test: /kaggle/input/datasets/ganeshiims/llm-finetune-dpo-dataset/dpo_test_clean.jsonl
    SFT adapter: /kaggle/input/datasets/ganeshiims/sft-lora/kaggle/working/sft_lora/checkpoint-303

 All paths verified!


In [9]:
# 2. LOAD JSONL DATA
def load_jsonl(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f'  Skipping line {i}: {e}')
    return rows


def validate_dpo_row(row, idx):
    """Check a single DPO row has all required fields and valid content."""
    errors = []
    for key in ['prompt', 'chosen', 'rejected']:
        if key not in row:
            errors.append(f'missing key: {key}')
        elif not isinstance(row[key], str):
            errors.append(f'{key} is not a string')
        elif not row[key].strip():
            errors.append(f'{key} is empty')
    if 'chosen' in row and 'rejected' in row:
        if row['chosen'] == row['rejected']:
            errors.append('chosen == rejected (no learning signal)')
    return errors


print(' Loading DPO datasets...')
raw_train = load_jsonl(DPO_TRAIN_FILE)
raw_val   = load_jsonl(DPO_VAL_FILE)
raw_test  = load_jsonl(DPO_TEST_FILE)

print(f'  Loaded → train: {len(raw_train)} | val: {len(raw_val)} | test: {len(raw_test)}')

# ── Validate all rows ──
print('\n Validating rows...')
valid_train, valid_val, valid_test = [], [], []
total_errors = 0

for split_name, raw, valid_list in [
    ('train', raw_train, valid_train),
    ('val',   raw_val,   valid_val),
    ('test',  raw_test,  valid_test),
]:
    errs = 0
    for i, row in enumerate(raw):
        issues = validate_dpo_row(row, i)
        if issues:
            errs += 1
            if errs <= 2:  # only print first 2
                print(f'  {split_name}[{i}]: {issues}')
        else:
            # Keep only the 3 required fields for TRL
            valid_list.append({
                'prompt':   row['prompt'],
                'chosen':   row['chosen'],
                'rejected': row['rejected'],
            })
    total_errors += errs
    print(f'  {split_name}: {len(valid_list)} valid, {errs} invalid')

print(f'\n  Total errors removed: {total_errors}')

# ── Convert to HuggingFace Dataset ──
train_dpo_dataset = Dataset.from_list(valid_train)
val_dpo_dataset   = Dataset.from_list(valid_val)
test_dpo_dataset  = Dataset.from_list(valid_test)

print(f'\n DPO datasets ready:')
print(f'   train_dpo_dataset: {len(train_dpo_dataset)} examples')
print(f'   val_dpo_dataset:   {len(val_dpo_dataset)} examples')
print(f'   test_dpo_dataset:  {len(test_dpo_dataset)} examples')
print(f'   Columns: {train_dpo_dataset.column_names}')


 Loading DPO datasets...
  Loaded → train: 1615 | val: 202 | test: 202

 Validating rows...
  train[477]: ['rejected is empty']
  train[822]: ['rejected is empty']
  train: 1612 valid, 3 invalid
  val[141]: ['rejected is empty']
  val: 201 valid, 1 invalid
  test: 202 valid, 0 invalid

  Total errors removed: 4

 DPO datasets ready:
   train_dpo_dataset: 1612 examples
   val_dpo_dataset:   201 examples
   test_dpo_dataset:  202 examples
   Columns: ['prompt', 'chosen', 'rejected']


In [10]:
# 3. Preview the DPO Data
print(' Sample DPO pairs from your dataset:')
print('=' * 70)
for i in range(min(5, len(train_dpo_dataset))):
    row = train_dpo_dataset[i]
    print(f'\n── Example {i+1} ──')
    print(f'  PROMPT:   {row["prompt"]}')
    print(f'  CHOSEN:   {row["chosen"]}')
    print(f'  REJECTED: {row["rejected"]}')
    # Show what's different
    try:
        c = json.loads(row['chosen'])
        r = json.loads(row['rejected'])
        c_keys = set(c.keys())
        r_keys = set(r.keys())
        if c_keys != r_keys:
            print(f'  DIFF:     keys differ — missing in rejected: {c_keys - r_keys}, extra: {r_keys - c_keys}')
        else:
            diffs = {k: (c[k], r[k]) for k in c_keys if c.get(k) != r.get(k)}
            if diffs:
                print(f'  DIFF:     value mismatch → {diffs}')
    except:
        print(f'  DIFF:     rejected is not valid JSON (intended)')

print('\n' + '=' * 70)
print(' If chosen is correct and rejected has a clear flaw → data is good!')


 Sample DPO pairs from your dataset:

── Example 1 ──
  PROMPT:   Aaron Mehta is a 37-year-old civil engineer living in Lima.
  CHOSEN:   {"name": "Aaron Mehta", "age": 37, "job": "civil engineer", "city": "Lima"}
  REJECTED: Extraction result: {"name": "Aaron Mehta", "age": 37, "job": "civil engineer", "city": "Lima"}. Hope this helps!
  DIFF:     rejected is not valid JSON (intended)

── Example 2 ──
  PROMPT:   Customer Zara Ali placed order #52914 for 10 units of green mechanical keyboard at $1544.96/unit. Total: $15449.6.
  CHOSEN:   {"customer": "Zara Ali", "order_id": 52914, "product": "mechanical keyboard", "color": "green", "quantity": 10, "price_per_unit": 1544.96, "total": 15449.6}
  REJECTED: {"customer": "Zara Ali", "order_id": 52914, "product": "N/A", "color": "green", "quantity": 10, "price_per_unit": null, "total": 15449.6}
  DIFF:     value mismatch → {'price_per_unit': (1544.96, None), 'product': ('mechanical keyboard', 'N/A')}

── Example 3 ──
  PROMPT:   I am Aarav 

In [11]:
# 4. Load Model + SFT Adapter
print(' Setting up 4-bit quantization config...')
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f' Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token     = tokenizer.eos_token
tokenizer.add_bos_token = False
print(f'   Vocab size: {tokenizer.vocab_size}')

# Print VRAM before loading
for i in range(torch.cuda.device_count()):
    free = (torch.cuda.get_device_properties(i).total_memory
            - torch.cuda.memory_reserved(i)) / 1e9
    print(f'   GPU {i} free before load: {free:.1f} GB')

print(f'\n Loading base model: {MODEL_NAME}')
model_config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)
model_config.torch_dtype = torch.float16

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    config=model_config,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.config.pad_token_id = tokenizer.eos_token_id
print('    Base model loaded')

print(f'\n Loading SFT adapter from: {SFT_ADAPTER}')
model = PeftModel.from_pretrained(model, SFT_ADAPTER, is_trainable=True)

# Cast LoRA params to float32 for stable gradients during DPO
lora_params = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)
        lora_params += param.numel()

print(f' SFT adapter loaded')
print(f'   Trainable LoRA params: {lora_params:,}')
print('\n Model ready for DPO training')
print('   ref_model=None → TRL will use frozen copy of base as reference (saves VRAM)')

 Setting up 4-bit quantization config...
 Loading tokenizer: Qwen/Qwen2.5-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

    Base model loaded

 Loading SFT adapter from: /kaggle/input/datasets/ganeshiims/sft-lora/kaggle/working/sft_lora/checkpoint-303
 SFT adapter loaded
   Trainable LoRA params: 40,370,176

 Model ready for DPO training
   ref_model=None → TRL will use frozen copy of base as reference (saves VRAM)


In [ ]:
# import inspect
# from trl import DPOConfig
# print(list(inspect.signature(DPOConfig.__init__).parameters.keys()))

In [12]:
# 5. DPO Training Config
torch.backends.cuda.matmul.allow_tf32 = True

dpo_config = DPOConfig(
    output_dir=DPO_DIR,
    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    warmup_steps=30,
    learning_rate=5e-5,
    weight_decay=0.01,
    beta=0.1,                 # DPO temperature — controls how strongly to prefer chosen

    fp16=False,
    bf16=False,

    logging_steps=5,
    eval_strategy='steps',
    eval_steps=25,
    save_strategy='steps',
    save_steps=25,
    save_total_limit=3,

    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='none',
    optim='paged_adamw_8bit',

    max_length=512,
    truncation_mode='keep_start',

    ddp_find_unused_parameters=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

print(' DPO config ready')
print(f'   Effective batch size: {dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps}')
print(f'   Epochs: {dpo_config.num_train_epochs}')
print(f'   LR: {dpo_config.learning_rate}')
print(f'   Beta: {dpo_config.beta}')
print(f'   Max length: {dpo_config.max_length}')


 DPO config ready
   Effective batch size: 16
   Epochs: 2
   LR: 5e-05
   Beta: 0.1
   Max length: 512


In [13]:
# Create DPO Trainer
dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,           # TRL uses a frozen copy of model as reference (saves VRAM)
    args=dpo_config,
    train_dataset=train_dpo_dataset,
    eval_dataset=val_dpo_dataset,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=3,   # stop after 3 evals with no improvement
        early_stopping_threshold=0.0
    )]
)

print(' DPO Trainer ready')
print(f'   Training examples: {len(train_dpo_dataset)}')
print(f'   Validation examples: {len(val_dpo_dataset)}')

# Estimate steps
steps_per_epoch = len(train_dpo_dataset) // (dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps)
total_steps = steps_per_epoch * dpo_config.num_train_epochs
print(f'   Steps per epoch: ~{steps_per_epoch}')
print(f'   Total steps: ~{total_steps}')


Adding EOS to train dataset:   0%|          | 0/1612 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1612 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokeni

Adding EOS to eval dataset:   0%|          | 0/201 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/201 [00:00<?, ? examples/s]

[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokenized prompt and the start of tokenized prompt+rejected. This may be due to unexpected tokenizer behavior, whitespace issues, or special token handling. Verify that the tokenizer is processing text consistently.
[RANK 0] Mismatch between tokeni

 DPO Trainer ready
   Training examples: 1612
   Validation examples: 201
   Steps per epoch: ~100
   Total steps: ~200


In [15]:
# Train
checkpoint = get_last_checkpoint(DPO_DIR)

if checkpoint is not None:
    print(f' Resuming from checkpoint: {checkpoint}')
    train_result = dpo_trainer.train(resume_from_checkpoint=checkpoint)
else:
    print('🚀 Starting DPO training from scratch...')
    print('   Watch for: rewards/chosen going UP, rewards/rejected going DOWN')
    print('   That is the DPO learning signal working correctly.\n')
    train_result = dpo_trainer.train()

print('\n DPO Training Finished!')
print(f'   Training loss: {train_result.training_loss:.4f}')

 Resuming from checkpoint: /kaggle/working/dpo_lora/checkpoint-25


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
50,0.018578,0.006866,0.117278,156426.000000,-0.460824,-0.597105,0.938762,0.218832,-10.694387,1.000000,10.912984,-26.102729,-172.695585
75,0.003199,0.004177,0.117272,213237.000000,-0.969813,-1.159491,0.936663,0.104431,-11.568948,1.000000,11.673313,-27.246735,-181.443719
100,0.001983,0.003250,0.116408,269794.000000,-1.211523,-1.422799,0.934425,0.029709,-12.200540,1.000000,12.230216,-27.993937,-187.759484
125,0.001590,0.002895,0.116064,324934.000000,-1.375801,-1.602692,0.928559,-0.144417,-12.627835,1.000000,12.483637,-29.735347,-192.035292
150,0.001983,0.002537,0.093948,380612.000000,-1.451520,-1.680731,0.924456,-0.155350,-12.812604,1.000000,12.657207,-29.844877,-193.881530
175,0.000847,0.002432,0.091023,437375.000000,-1.484358,-1.714569,0.923754,-0.156500,-12.860862,1.000000,12.704456,-29.855955,-194.363029
200,0.000670,0.002443,0.091465,495532.000000,-1.487560,-1.718399,0.924388,-0.148374,-12.860079,1.000000,12.711968,-29.774953,-194.355255
202,0.000670,0.002439,0.091583,499507.000000,-1.487359,-1.718219,0.924302,-0.147969,-12.860643,1.000000,12.712837,-29.770756,-194.359453



 DPO Training Finished!
   Training loss: 0.0036


In [16]:
# Save the final best model
final_path = os.path.join(DPO_DIR, 'final_adapter')
dpo_trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

print(f' DPO adapter saved to: {final_path}')
print(f'   Files: {os.listdir(final_path)}')

# Save training metrics
metrics = train_result.metrics
with open(os.path.join(DPO_DIR, 'dpo_train_metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'\n Training metrics:')
for k, v in metrics.items():
    print(f'   {k}: {v}')

 DPO adapter saved to: /kaggle/working/dpo_lora/final_adapter
   Files: ['adapter_model.safetensors', 'tokenizer_config.json', 'README.md', 'adapter_config.json', 'training_args.bin', 'ref', 'chat_template.jinja', 'tokenizer.json']

 Training metrics:
   train_runtime: 5904.6721
   train_samples_per_second: 0.546
   train_steps_per_second: 0.034
   total_flos: 2.093789333615616e+16
   train_loss: 0.003647614528637121
   epoch: 2.0


In [17]:
# Test a few prompts to see if the DPO model is better than SFT.
model.eval()

test_prompts = [
    'Yusuf Martinez, 38, software engineer at TCS in Jakarta.',
    'Order #58448: 3 x smartwatch at $1434.76 each. Total: $4304.28.',
    'Schedule a design review on 08/05 at 11:00 in Room A.',
    'Hello, how are you?',  # refusal case
]

instruction = 'Extract the key information from the text and return it as JSON.'

print(' DPO Model Inference Test')
print('=' * 60)

for prompt in test_prompts:
    full_input = f'### Instruction:\n{instruction}\n\n### Input:\n{prompt}\n\n### Response:\n'
    inputs = tokenizer(full_input, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Check if output is valid JSON
    try:
        parsed = json.loads(response)
        json_valid = ' valid JSON'
    except:
        json_valid = ' not valid JSON'

    print(f'\nINPUT:    {prompt}')
    print(f'OUTPUT:   {response}')
    print(f'STATUS:   {json_valid}')


 DPO Model Inference Test

INPUT:    Yusuf Martinez, 38, software engineer at TCS in Jakarta.
OUTPUT:   {"name": "Yusuf Martinez", "age": 38, "job": "software engineer", "company": "TCS", "city": "Jakarta"}
STATUS:    valid JSON

INPUT:    Order #58448: 3 x smartwatch at $1434.76 each. Total: $4304.28.
OUTPUT:   {"order_id": 58448, "product": "smartwatch", "quantity": 3, "price_per_unit": 1434.76, "total": 4304.28}
STATUS:    valid JSON

INPUT:    Schedule a design review on 08/05 at 11:00 in Room A.
OUTPUT:   {"event": "design review", "date": "08/05", "time": "11:00", "location": "Room A"}
STATUS:    valid JSON

INPUT:    Hello, how are you?
OUTPUT:   {}
STATUS:    valid JSON


In [35]:
# Test a few prompts to see if the DPO model is better than SFT.
model.eval()

test_prompts = [
    # easy
    "John is 25 years old and lives in London.",
    
    # medium
    "Sarah, age 41, works at Google in Paris. Email: sarah@gmail.com, phone: +33 12345",
    
    # hard
    "Yesterday evening around 5ish, Ahmed called regarding INV-9982 saying payment maybe delayed...",
    
    # edge
    "Hello",
    "Tell me a joke",
]
instruction = """
Extract the key information from the text and return ONLY valid JSON.
Do not include explanations, notes, or extra text.
"""

print(' DPO Model Inference Test')
print('=' * 60)

for prompt in test_prompts:
    full_input = f'### Instruction:\n{instruction}\n\n### Input:\n{prompt}\n\n### Response:\n'
    inputs = tokenizer(full_input, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    # Check if output is valid JSON
    try:
        parsed = json.loads(response)
        json_valid = ' valid JSON'
    except:
        json_valid = ' not valid JSON'

    print(f'\nINPUT:    {prompt}')
    print(f'OUTPUT:   {response}')
    print(f'STATUS:   {json_valid}')


 DPO Model Inference Test

INPUT:    John is 25 years old and lives in London.
OUTPUT:   {"name": "John", "age": 25, "city": "London"}
STATUS:    valid JSON

INPUT:    Sarah, age 41, works at Google in Paris. Email: sarah@gmail.com, phone: +33 12345
OUTPUT:   {"name": "Sarah", "age": 41, "job": "worker", "company": "Google", "city": "Paris", "email": "sarah@gmail.com", "phone": "+33 12345"}
STATUS:    valid JSON

INPUT:    Yesterday evening around 5ish, Ahmed called regarding INV-9982 saying payment maybe delayed...
OUTPUT:   {"customer": "Ahmed", "time": "yesterday evening 5ish", "event": "called", "issue": "payment maybe delayed", "order_id": 9982}
STATUS:    valid JSON

INPUT:    Hello
OUTPUT:   {}
STATUS:    valid JSON

INPUT:    Tell me a joke
OUTPUT:   {}
STATUS:    valid JSON


In [18]:
# Evaluate on Test Set
print(' Evaluating DPO model on test set...')
print('(This may take a few minutes)\n')

model.eval()
instruction = 'Extract the key information from the text and return it as JSON.'

valid_json_count = 0
exact_match_count = 0
field_recall_sum = 0
total = 0
errors = []

# Evaluate on first 100 test examples (full eval takes too long on free GPU)
eval_subset = valid_test[:100]

for ex in eval_subset:
    prompt    = ex['prompt']
    chosen    = ex['chosen']   # ground truth

    full_input = f'### Instruction:\n{instruction}\n\n### Input:\n{prompt}\n\n### Response:\n'
    inputs = tokenizer(full_input, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    pred = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    total += 1

    # Metric 1: JSON validity
    try:
        pred_dict = json.loads(pred)
        valid_json_count += 1

        # Metric 2: Exact match
        truth_dict = json.loads(chosen)
        if pred_dict == truth_dict:
            exact_match_count += 1

        # Metric 3: Field recall
        truth_keys = set(truth_dict.keys())
        pred_keys  = set(pred_dict.keys())
        if truth_keys:
            recall = len(truth_keys & pred_keys) / len(truth_keys)
            field_recall_sum += recall

    except Exception as e:
        errors.append((prompt[:50], pred[:50], str(e)))

# Results
json_validity  = valid_json_count / total * 100
exact_match    = exact_match_count / total * 100
field_recall   = field_recall_sum / total * 100

print(' DPO MODEL EVALUATION RESULTS')
print('=' * 50)
print(f'  Evaluated on:      {total} test examples')
print(f'  JSON Validity:     {json_validity:.1f}%')
print(f'  Exact Match:       {exact_match:.1f}%')
print(f'  Field Recall:      {field_recall:.1f}%')
print(f'  Invalid JSON:      {len(errors)}')

# Save results
results = {
    'model': 'DPO',
    'test_examples': total,
    'json_validity_pct': round(json_validity, 2),
    'exact_match_pct': round(exact_match, 2),
    'field_recall_pct': round(field_recall, 2),
}
with open(os.path.join(DPO_DIR, 'dpo_eval_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f'\n Results saved to: {DPO_DIR}/dpo_eval_results.json')
print('\n Compare these numbers against your SFT baseline to show improvement!')


 Evaluating DPO model on test set...
(This may take a few minutes)

 DPO MODEL EVALUATION RESULTS
  Evaluated on:      100 test examples
  JSON Validity:     98.0%
  Exact Match:       89.0%
  Field Recall:      94.8%
  Invalid JSON:      2

 Results saved to: /kaggle/working/dpo_lora/dpo_eval_results.json

 Compare these numbers against your SFT baseline to show improvement!


In [19]:
# import os
# import zipfile
# from IPython.display import FileLink

# # 1. Zip the folder
# os.system("zip -r dpo_lora.zip /kaggle/working/dpo_lora")

# # 2. Generate and display the download link
# FileLink(r'dpo_lora.zip')

# # !zip -r /kaggle/working/sft_lora.zip /kaggle/working/sft_lora

  adding: kaggle/working/dpo_lora/ (stored 0%)
  adding: kaggle/working/dpo_lora/checkpoint-202/ (stored 0%)
  adding: kaggle/working/dpo_lora/checkpoint-202/adapter_model.safetensors (deflated 21%)
  adding: kaggle/working/dpo_lora/checkpoint-202/rng_state.pth (deflated 26%)
  adding: kaggle/working/dpo_lora/checkpoint-202/tokenizer_config.json (deflated 60%)
  adding: kaggle/working/dpo_lora/checkpoint-202/trainer_state.json (deflated 78%)
  adding: kaggle/working/dpo_lora/checkpoint-202/README.md (deflated 65%)
  adding: kaggle/working/dpo_lora/checkpoint-202/adapter_config.json (deflated 59%)
  adding: kaggle/working/dpo_lora/checkpoint-202/scheduler.pt (deflated 62%)
  adding: kaggle/working/dpo_lora/checkpoint-202/training_args.bin (deflated 53%)
  adding: kaggle/working/dpo_lora/checkpoint-202/ref/ (stored 0%)
  adding: kaggle/working/dpo_lora/checkpoint-202/ref/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/dpo_lora/checkpoint-202/ref/adapter_config.json (defl

/kaggle/working/dpo_lora.zip

In [21]:
import os

target_dir = '/kaggle/working/dpo_lora'

print(f'Inspecting all levels of: {target_dir}')
if os.path.exists(target_dir):
    # Walk through the directory tree recursively
    for root, dirs, files in os.walk(target_dir):
        # Calculate current folder level relative to target path
        level = root.replace(target_dir, '').count(os.sep)
        indent = '  ' * level
        
        # Print the current directory name
        folder_name = os.path.basename(root)
        if folder_name:
            print(f'{indent}Folder: {folder_name}/')
            
        # Print all files inside this specific subfolder
        sub_indent = '  ' * (level + 1)
        for file in files:
            print(f'{sub_indent}File:   {file}')
            
    # Quick sanity check for completely empty root directory
    if not os.listdir(target_dir):
        print('  The directory is completely empty.')
else:
    print('  The directory does not exist yet.')


Inspecting all levels of: /kaggle/working/dpo_lora
Folder: dpo_lora/
  File:   dpo_eval_results.json
  File:   README.md
  File:   dpo_train_metrics.json
  Folder: checkpoint-202/
    File:   adapter_model.safetensors
    File:   rng_state.pth
    File:   tokenizer_config.json
    File:   trainer_state.json
    File:   README.md
    File:   adapter_config.json
    File:   scheduler.pt
    File:   training_args.bin
    File:   optimizer.pt
    File:   chat_template.jinja
    File:   tokenizer.json
    Folder: ref/
      File:   adapter_model.safetensors
      File:   adapter_config.json
  Folder: checkpoint-175/
    File:   adapter_model.safetensors
    File:   rng_state.pth
    File:   tokenizer_config.json
    File:   trainer_state.json
    File:   README.md
    File:   adapter_config.json
    File:   scheduler.pt
    File:   training_args.bin
    File:   optimizer.pt
    File:   chat_template.jinja
    File:   tokenizer.json
    Folder: ref/
      File:   adapter_model.safetensors
  